# Lada on Google Colab

Restore pixelated videos with the [Lada](https://codeberg.org/ladaapp/lada) CLI on a Colab GPU.

**How to use**

1. Make sure the notebook runs on a GPU: *Runtime → Change runtime type → T4 GPU* (any other NVIDIA GPU works as well).
2. Upload your videos to Google Drive, by default into the folder `MyDrive/lada/input`.
3. Adjust the settings below if needed and run all cells: *Runtime → Run all*. Colab will ask for permission to access your Google Drive.
4. Restored videos are saved to `MyDrive/lada/output` as `<name>.restored.mp4`.

**Good to know**

* Only the CLI is available on Colab. The GUI needs a desktop and can't run here, so there is no real-time preview. Videos are exported only.
* Lada is installed into its own virtual environment from `uv.lock`, so it gets exactly the dependency versions the project is tested with and doesn't conflict with the packages that come with Colab. Installation takes a few minutes and has to be repeated in every new Colab session.
* Colab sessions can disconnect at any time. Long videos are therefore split into parts (see `SPLIT_MINUTES`) and every restored part is saved to Drive right away. After a disconnect, run all cells again: videos and parts that are already done will be skipped.
* Make sure your usage complies with the policies of Google Colab and Google Drive.

In [ ]:
# @title Settings
# @markdown **Paths.** Google Drive is mounted at `/content/drive`. `INPUT_PATH` can be a folder or a single video file.
INPUT_PATH = "/content/drive/MyDrive/lada/input"  # @param {type:"string"}
OUTPUT_DIR = "/content/drive/MyDrive/lada/output"  # @param {type:"string"}
# @markdown **Models.** The defaults are the ones recommended by Lada.
DETECTION_MODEL = "v4-fast"  # @param ["v4-fast", "v4-accurate"]
RESTORATION_MODEL = "basicvsrpp-v1.2"  # @param ["basicvsrpp-v1.2", "deepmosaics"]
# @markdown **Export.** `auto` uses the Nvidia hardware encoder if the GPU has one (T4, L4) and the CPU encoder otherwise (A100, H100). `nvidia` presets only work on GPUs with a hardware encoder. `h264-nvidia-gpu-hq` is added by this notebook for H.264 in high quality.
ENCODING_PRESET = "auto"  # @param ["auto", "hevc-nvidia-gpu-hq", "hevc-nvidia-gpu-balanced", "hevc-nvidia-gpu-uhq", "h264-nvidia-gpu-hq", "h264-nvidia-gpu-fast", "h264-cpu-fast", "h264-cpu-uhq", "av1-cpu-uhq"]
# @markdown **Memory.** Maximum number of frames restored at once. Lower it (e.g. 90) if you run out of RAM or VRAM.
MAX_CLIP_LENGTH = 180  # @param {type:"integer"}
# @markdown **Splitting.** Length of the parts long videos are split into, so progress survives a disconnect. 0 restores each video in one go.
SPLIT_MINUTES = 10  # @param {type:"integer"}
# @markdown **Source code**
REPO_URL = "https://github.com/antorio/lada.git"  # @param {type:"string"}
REPO_BRANCH = "main"  # @param {type:"string"}

import codecs
import os
import signal
import subprocess
import sys

REPO_DIR = "/content/lada"
WORK_DIR = "/content/lada_work"

def lada_env() -> dict:
    """Environment for commands run in Lada's virtual environment, without settings of the Colab kernel leaking into it."""
    env = dict(os.environ,
               LADA_REPO_URL=REPO_URL, LADA_REPO_BRANCH=REPO_BRANCH, LADA_REPO_DIR=REPO_DIR,
               LADA_DETECTION_MODEL=DETECTION_MODEL, LADA_RESTORATION_MODEL=RESTORATION_MODEL,
               LADA_MODEL_WEIGHTS_DIR=f"{REPO_DIR}/model_weights",
               MPLBACKEND="Agg", PYTHONUNBUFFERED="1", COLUMNS="120", LINES="40")
    env.pop("PYTHONPATH", None)
    return env

def run(cmd: list[str], **kwargs) -> None:
    """Run a command and stream its output, progress bars included, into the cell output."""
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, **kwargs)
    decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
    try:
        while chunk := process.stdout.read1(65536):
            sys.stdout.write(decoder.decode(chunk))
            sys.stdout.flush()
    except KeyboardInterrupt:
        # Let the command shut down cleanly when the cell gets interrupted
        process.send_signal(signal.SIGINT)
        process.wait()
        raise
    if process.wait() != 0:
        raise subprocess.CalledProcessError(process.returncode, cmd)

In [ ]:
# @title Check GPU and mount Google Drive
from pathlib import Path

try:
    run(["nvidia-smi"])
except (FileNotFoundError, subprocess.CalledProcessError):
    raise RuntimeError("No NVIDIA GPU found. Select one via Runtime → Change runtime type → T4 GPU and run all cells again.")

if INPUT_PATH.startswith("/content/drive") or OUTPUT_DIR.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
if not Path(INPUT_PATH).exists():
    if Path(INPUT_PATH).suffix:
        print(f"WARNING: Input video {INPUT_PATH} does not exist")
    else:
        Path(INPUT_PATH).mkdir(parents=True)
        print(f"Created input folder {INPUT_PATH}. Upload your videos there before running the restore cell.")

In [ ]:
# @title Install Lada
# @markdown Clones the source code, installs the dependencies from `uv.lock` into `/content/lada/.venv`, applies Lada's patches to its dependencies and downloads the model weights.
INSTALL_SCRIPT = r'''
if [ ! -d "$LADA_REPO_DIR/.git" ]; then
  echo "==> Cloning $LADA_REPO_URL ($LADA_REPO_BRANCH)"
  git clone --quiet --depth 1 --branch "$LADA_REPO_BRANCH" "$LADA_REPO_URL" "$LADA_REPO_DIR"
fi
cd "$LADA_REPO_DIR"

echo "==> Installing Python dependencies"
pip install --quiet uv
UV_LINK_MODE=copy uv sync --frozen --extra nvidia

echo "==> Applying patches"
site_packages=$(.venv/bin/python -c "import sysconfig; print(sysconfig.get_paths()['purelib'])")
for p in increase_mms_time_limit.patch remove_ultralytics_telemetry.patch fix_loading_mmengine_weights_on_torch26_and_higher.diff; do
  # Cells are often run more than once, so skip patches that have already been applied
  if patch -R -p1 -s -f --dry-run -d "$site_packages" -i "$PWD/patches/$p" > /dev/null 2>&1; then
    echo "    $p (already applied)"
  else
    patch -p1 -s -f -d "$site_packages" -i "$PWD/patches/$p"
    echo "    $p"
  fi
done

echo "==> Adding encoding preset h264-nvidia-gpu-hq"
# Lada comes without a high quality H.264 preset for Nvidia GPUs, so add one based on hevc-nvidia-gpu-hq
presets=$(.venv/bin/python -c "import importlib.util, os; print(os.path.join(importlib.util.find_spec('lada').submodule_search_locations[0], 'utils', 'encoding_presets.csv'))")
if ! grep -q "^h264-nvidia-gpu-hq|" "$presets"; then
  echo "h264-nvidia-gpu-hq|H.264 / AVC, Nvidia hardware encoder, High Quality, Large File Size|h264_nvenc|-preset p7 -tune hq -profile high -rc constqp -qp 18 -spatial_aq 1 -aq-strength 6 -bf 3 -b_ref_mode middle -rc-lookahead 32" >> "$presets"
fi

echo "==> Downloading model weights"
cd model_weights
download() {
  if [ ! -s "$2" ]; then
    echo "    $2"
    wget --quiet "$1" -O "$2.part"
    mv "$2.part" "$2"
  fi
}
hf=https://huggingface.co/ladaapp/lada/resolve/main
download "$hf/lada_mosaic_detection_model_v4_fast.pt?download=true" lada_mosaic_detection_model_v4_fast.pt
download "$hf/lada_mosaic_restoration_model_generic_v1.2.pth?download=true" lada_mosaic_restoration_model_generic_v1.2.pth
if [ "$LADA_DETECTION_MODEL" = "v4-accurate" ]; then
  download "$hf/lada_mosaic_detection_model_v4_accurate.pt?download=true" lada_mosaic_detection_model_v4_accurate.pt
fi
if [ "$LADA_RESTORATION_MODEL" = "deepmosaics" ]; then
  download "https://drive.usercontent.google.com/download?id=1ulct4RhRxQp1v5xwEmUH7xz7AK42Oqlw&export=download&confirm=t" 3rd_party/clean_youknow_video.pth
fi
if ! sha256sum --check --ignore-missing --quiet checksums_sha256.txt; then
  echo "Checksum mismatch: delete the files marked FAILED from $PWD and run this cell again"
  exit 1
fi
cd ..

echo "==> Checking installation"
.venv/bin/python -c "import torch; assert torch.cuda.is_available(), 'CUDA is not available'; print('PyTorch', torch.__version__, 'on', torch.cuda.get_device_name(0))"
.venv/bin/lada-cli --version
.venv/bin/lada-cli --list-encoding-presets
'''
run(["bash", "-euo", "pipefail", "-c", INSTALL_SCRIPT], env=lada_env())

In [ ]:
# @title Restore videos
# @markdown Restores the videos from `INPUT_PATH` and saves them to `OUTPUT_DIR`. Videos that already have a restored file in `OUTPUT_DIR` are skipped.
import contextlib
import shutil
from pathlib import Path

LADA_CLI = f"{REPO_DIR}/.venv/bin/lada-cli"
VIDEO_EXTENSIONS = {".mp4", ".m4v", ".mkv", ".mov", ".avi", ".wmv", ".webm", ".flv", ".ts", ".mpg", ".mpeg"}

def is_done(path: Path) -> bool:
    return path.is_file() and path.stat().st_size > 0

def publish(src: Path, dest: Path):
    """Move a finished file to its destination (usually on Drive) in a way that never leaves a half-written file behind."""
    dest.parent.mkdir(parents=True, exist_ok=True)
    partial = dest.with_name(dest.name + ".part")
    shutil.copyfile(src, partial)
    os.replace(partial, dest)
    src.unlink()

def ffmpeg(*args: str):
    run(["ffmpeg", "-hide_banner", "-loglevel", "error", "-y", *args])

def lada_restore(src: Path, dest: Path, work_dir: Path) -> bool:
    restored = work_dir / "restored" / dest.name
    cmd = [LADA_CLI, "--input", str(src), "--output", str(restored), "--temporary-directory", str(work_dir / "tmp"),
           "--mosaic-detection-model", DETECTION_MODEL, "--mosaic-restoration-model", RESTORATION_MODEL,
           "--max-clip-length", str(MAX_CLIP_LENGTH)]
    if ENCODING_PRESET != "auto":
        cmd += ["--encoding-preset", ENCODING_PRESET]
    run(cmd, cwd=REPO_DIR, env=lada_env())
    # lada-cli doesn't exit with an error code if the export fails, so check for the restored file instead
    if not is_done(restored):
        print(f"Restoring {src.name} failed, see the output above")
        return False
    publish(restored, dest)
    return True

def split_video(src: Path, parts_dir: Path) -> list[Path]:
    # Cuts happen at keyframes without re-encoding, so the same video is always split into the same parts
    parts_dir.mkdir(parents=True)
    ffmpeg("-i", str(src), "-map", "0:v:0", "-map", "0:a?", "-c", "copy",
           "-f", "segment", "-segment_time", str(SPLIT_MINUTES * 60), "-reset_timestamps", "1",
           str(parts_dir / f"part%03d{src.suffix.lower()}"))
    return sorted(parts_dir.iterdir())

def join_parts(parts: list[Path], dest: Path):
    file_list = dest.with_suffix(".txt")
    file_list.write_text("".join("file '{}'\n".format(str(part).replace("'", "'\\''")) for part in parts))
    ffmpeg("-f", "concat", "-safe", "0", "-i", str(file_list), "-map", "0", "-c", "copy", "-movflags", "+faststart", str(dest))

def restore_video(src: Path) -> bool:
    dest = Path(OUTPUT_DIR) / f"{src.stem}.restored.mp4"
    if is_done(dest):
        print(f"{src.name}: already restored, skipping")
        return True
    work_dir = Path(WORK_DIR) / src.stem
    shutil.rmtree(work_dir, ignore_errors=True)
    (work_dir / "tmp").mkdir(parents=True)
    try:
        if SPLIT_MINUTES <= 0:
            print(f"{src.name}: copying to local disk")
            local_copy = work_dir / src.name
            shutil.copyfile(src, local_copy)
            print(f"{src.name}: restoring")
            return lada_restore(local_copy, dest, work_dir)

        print(f"{src.name}: splitting into parts of {SPLIT_MINUTES} min")
        parts = split_video(src, work_dir / "parts")
        # Restored parts stay on Drive until the whole video is done so they survive a disconnect
        restored_parts_dir = Path(OUTPUT_DIR) / ".lada_parts" / f"{src.stem}.{SPLIT_MINUTES}min"
        restored_parts = []
        for i, part in enumerate(parts, start=1):
            restored_part = restored_parts_dir / f"{part.stem}.restored.mp4"
            if is_done(restored_part):
                print(f"{src.name}: part {i}/{len(parts)} already restored, skipping")
            else:
                print(f"{src.name}: restoring part {i}/{len(parts)}")
                if not lada_restore(part, restored_part, work_dir):
                    return False
            restored_parts.append(restored_part)

        print(f"{src.name}: joining parts")
        joined = work_dir / dest.name
        join_parts(restored_parts, joined)
        publish(joined, dest)
        shutil.rmtree(restored_parts_dir)
        with contextlib.suppress(OSError):
            restored_parts_dir.parent.rmdir()
        return True
    finally:
        shutil.rmtree(work_dir, ignore_errors=True)

if not Path(LADA_CLI).exists():
    raise RuntimeError("Lada is not installed yet. Run the cell 'Install Lada' first.")

input_path = Path(INPUT_PATH)
if input_path.is_file():
    videos = [input_path]
elif input_path.is_dir():
    videos = sorted(p for p in input_path.iterdir() if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS)
else:
    raise FileNotFoundError(f"INPUT_PATH does not exist: {INPUT_PATH}")
if not videos:
    raise FileNotFoundError(f"No videos found in {INPUT_PATH}")

print(f"Found {len(videos)} video(s)\n")
failed = []
for video in videos:
    if not restore_video(video):
        failed.append(video.name)
    print()
print(f"Finished. Restored videos are in {OUTPUT_DIR}")
if failed:
    print("Failed:", ", ".join(failed))

In [ ]:
# @title List restored videos
for path in sorted(Path(OUTPUT_DIR).glob("*.restored.mp4")):
    print(f"{path.stat().st_size / 2**30:7.2f} GB  {path.name}")

## Troubleshooting

* **Session disconnected or runtime restarted:** Run all cells again. Lada gets installed again, but videos and parts that are already restored will be skipped.
* **Out of memory** (CUDA out of memory or the session crashes): Lower `MAX_CLIP_LENGTH`, e.g. to 90 or 60. 4K videos need a lot more RAM than the free tier offers.
* **Export is slow on A100 or H100:** These GPUs have no hardware video encoder, so videos are encoded on the CPU.
* **Small glitches where parts are joined:** Parts are cut at keyframes and restored independently. Set `SPLIT_MINUTES` to 0 to restore videos in one go, at the cost of starting over after a disconnect.
* **Changed models in the middle of a video:** Parts restored with the previous settings are kept in `OUTPUT_DIR/.lada_parts/`. Delete the video's folder there if you want all parts restored with the new settings.
* **More options:** Run `run([LADA_CLI, "--help"], cwd=REPO_DIR, env=lada_env())` in a new cell to see all options of the CLI.